In [1]:
import os
from pathlib import Path
import uuid

import pandas as pd
import numpy as np
from unidecode import unidecode
import re

In [2]:

# Paths
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TRAINING_PATH = OUTPUT_DIR / "training_data_v2.csv"
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"

MASTER_TEAM_LIST_PATH = Path("master_team_list.csv")

print("📂 DATA_ROOT:", DATA_ROOT.resolve())
print("📂 OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("📄 training_data_v2 path:", TRAINING_PATH)


📂 DATA_ROOT: C:\Python\fpl_pipeline\data
📂 OUTPUT_DIR: C:\Python\fpl_pipeline\output
📄 training_data_v2 path: output\training_data_v2.csv


In [3]:
def normalize_player_name(name: str) -> str:
    """
    Normalizes player names:
    - to lowercase
    - remove accents
    - strip trailing numbers (e.g. ' 534')
    - remove underscores
    - keep only alphanumeric + spaces
    - collapse multiple spaces
    """
    if pd.isna(name):
        return name

    name = str(name).strip().lower()
    name = unidecode(name)

    # Remove trailing numeric suffixes: "aaron connolly 534", "aaron_connolly_534"
    name = re.sub(r'[\s_]*\d+\s*$', '', name)

    # Replace underscores with spaces
    name = name.replace("_", " ")

    # Keep only alphanumeric + space
    name = "".join(c for c in name if c.isalnum() or c.isspace())

    # Collapse multiple spaces
    name = " ".join(name.split())

    return name


# Quick sanity test
print("🔎 Testing normalize_player_name:")
for t in ["aaron cresswell 376", "Aaron_Connolly534", "Son Heung-Min 123"]:
    print(f"  '{t}' -> '{normalize_player_name(t)}'")


🔎 Testing normalize_player_name:
  'aaron cresswell 376' -> 'aaron cresswell'
  'Aaron_Connolly534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'


In [4]:
def load_all_gws(data_root=DATA_ROOT):
    all_seasons = []
    print("\n📥 Loading all GW data...")

    for season in sorted(os.listdir(data_root)):
        season_path = data_root / season
        gws_path = season_path / "gws"

        if not season_path.is_dir():
            continue
        if not gws_path.exists():
            continue

        gw_files = sorted(
            [f for f in os.listdir(gws_path) if f.startswith("gw") and f.endswith(".csv")],
            key=lambda x: int(x.replace("gw","").replace(".csv",""))
        )

        print(f"  → Season {season}: {len(gw_files)} GWs")

        frames = []
        for fname in gw_files:
            gw = int(fname.replace("gw","").replace(".csv",""))
            df = pd.read_csv(gws_path / fname)

            keep = [
                "name", "element", "minutes", "goals_scored", "assists", "clean_sheets",
                "goals_conceded","yellow_cards","red_cards","total_points",
                "influence","creativity","threat","ict_index",
                "opponent_team","was_home"
            ]
            cols = [c for c in keep if c in df.columns]
            df = df[cols].copy()

            df["season"] = season
            df["Gameweek"] = gw
            frames.append(df)

        all_seasons.append(pd.concat(frames, ignore_index=True))

    df_gws = pd.concat(all_seasons, ignore_index=True)
    print(f"✅ GW rows: {len(df_gws):,}")
    return df_gws


df_gws = load_all_gws()
df_gws.head()



📥 Loading all GW data...
  → Season 2018-19: 38 GWs
  → Season 2019-20: 38 GWs
  → Season 2020-21: 38 GWs


C:\Users\SOFI\AppData\Local\Temp\ipykernel_6452\3321161310.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


  → Season 2021-22: 38 GWs
  → Season 2022-23: 38 GWs


C:\Users\SOFI\AppData\Local\Temp\ipykernel_6452\3321161310.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


  → Season 2023-24: 38 GWs
  → Season 2024-25: 38 GWs
✅ GW rows: 171,993


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,influence,creativity,threat,ict_index,opponent_team,was_home,season,Gameweek
0,Aaron_Cresswell_402,402,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,12,False,2018-19,1
1,Aaron_Lennon_83,83,90,0,0,1,0,0,0,3,10.0,12.3,17.0,3.9,16,False,2018-19,1
2,Aaron_Mooy_199,199,90,0,0,0,3,0,0,2,20.2,18.2,0.0,3.8,6,True,2018-19,1
3,Aaron_Ramsey_14,14,53,0,0,0,1,0,0,1,9.4,10.8,9.0,2.9,13,True,2018-19,1
4,Aaron_Wan-Bissaka_145,145,90,0,1,1,0,0,0,12,46.0,14.0,0.0,6.0,9,False,2018-19,1


In [5]:
def load_players_raw_by_season():
    players = {}
    print("\n📥 Loading players_raw per season...")

    for season in sorted(df_gws["season"].unique()):
        path = DATA_ROOT / season / "players_raw.csv"
        if not path.exists():
            print(f"  ⚠️ No players_raw for {season}")
            continue

        df = pd.read_csv(path)
        if "id" in df.columns:
            df.rename(columns={"id":"element"}, inplace=True)

        keep = ["element","team","element_type","web_name","first_name","second_name"]
        df = df[[c for c in keep if c in df.columns]].copy()

        df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
        df["team"] = pd.to_numeric(df["team"], errors="coerce").astype("Int64")

        players[season] = df.set_index("element")
        print(f"  → {season}: {len(df)} players")

    return players


players_raw_by_season = load_players_raw_by_season()


📥 Loading players_raw per season...
  → 2018-19: 624 players
  → 2019-20: 666 players
  → 2020-21: 713 players
  → 2021-22: 737 players
  → 2022-23: 778 players
  → 2023-24: 865 players
  → 2024-25: 804 players


In [6]:
master_teams_df = pd.read_csv(MASTER_TEAM_LIST_PATH) if MASTER_TEAM_LIST_PATH.exists() else None
if master_teams_df is not None:
    master_teams_df.columns = [c.lower() for c in master_teams_df.columns]

def load_teams_for_season(season: str):
    path = DATA_ROOT / season / "teams.csv"

    if path.exists():
        df = pd.read_csv(path)
        idcol = "id" if "id" in df.columns else "code"
        df.rename(columns={idcol:"Team ID", "name":"Team Name"}, inplace=True)
        df["Team ID"] = pd.to_numeric(df["Team ID"], errors="coerce").astype("Int64")
        if "short_name" not in df.columns:
            df["short_name"] = df["Team Name"]
        return df[["Team ID","Team Name","short_name"]]

    if master_teams_df is not None:
        sub = master_teams_df[master_teams_df["season"] == season].copy()
        if not sub.empty:
            sub.rename(columns={"team":"Team ID","team_name":"Team Name"}, inplace=True)
            sub["Team ID"] = pd.to_numeric(sub["Team ID"], errors="coerce").astype("Int64")
            sub["short_name"] = sub["Team Name"]
            print(f"  🔁 Using master_team_list for {season}")
            return sub[["Team ID","Team Name","short_name"]]

    raise RuntimeError(f"❌ No team info for {season}")

In [7]:
# %% [markdown]
# ## 4. Build core training table (matching to fixtures WITHOUT fixture id)

# %%
import numpy as np

print("📦 Building core training table using (season, GW, team vs opponent) matching...")

df2 = df_gws.copy()

# Ensure numeric types
df2["Gameweek"] = pd.to_numeric(df2["Gameweek"], errors="coerce").astype("Int64")
df2["element"]  = pd.to_numeric(df2["element"], errors="coerce").astype("Int64")
df2["opponent_team"] = pd.to_numeric(df2["opponent_team"], errors="coerce").astype("Int64")

# Prepare fixtures
fx = fixtures.copy()
fx.rename(columns={"event":"Gameweek"}, inplace=True)
fx["Gameweek"] = pd.to_numeric(fx["Gameweek"], errors="coerce").astype("Int64")

# ------------------------------------------
# 1️⃣ Infer Player Team ID (from was_home)
# ------------------------------------------
# If was_home = True -> team = team_h, opponent = team_a
# If was_home = False -> team = team_a, opponent = team_h

df2["Player Team ID"] = np.where(
    df2["was_home"] == True,
    df2["team"],   # in your data "team" is the player's team for that gw
    df2["team"]    # but this may not exist → we reconstruct below
)

# Actually reconstruct:
df2["Player Team ID"] = np.where(
    df2["was_home"] == True,
    df2["team"],   # if exists
    df2["team"]    # fallback, will be fixed below
)

# If "team" column does not exist, build it:
if "team" not in df2.columns:
    # reconstruct team from players_raw
    print("🔧 Reconstructing Player Team ID from players_raw files...")
    df2["Player Team ID"] = df2["element"].map(
        players_raw_all.set_index("element")["team"]
    ).astype("Int64")

# Opponent:
df2["Opponent ID"] = df2["opponent_team"]

# ------------------------------------------
# 2️⃣ Merge with fixtures based on actual match logic
# ------------------------------------------

print("🔍 Matching rows to fixtures...")

merged = df2.merge(
    fx[[
        "season", "Gameweek",
        "team_h", "team_a",
        "team_h_difficulty", "team_a_difficulty"
    ]],
    left_on=["season", "Gameweek", "Player Team ID", "Opponent ID"],
    right_on=["season", "Gameweek", "team_h", "team_a"],
    how="left"
)

# Try reverse (player was away)
mask_missing = merged["team_h_difficulty"].isna()

merged2 = df2[mask_missing].merge(
    fx[[
        "season", "Gameweek",
        "team_h", "team_a",
        "team_h_difficulty", "team_a_difficulty"
    ]],
    left_on=["season", "Gameweek", "Player Team ID", "Opponent ID"],
    right_on=["season", "Gameweek", "team_a", "team_h"],
    how="left"
)

# Fill missing from reverse match
for col in merged.columns:
    merged.loc[mask_missing, col] = merged2[col].values

# ------------------------------------------
# 3️⃣ Assign difficulty correctly
# ------------------------------------------

merged["Is Home"] = merged["was_home"].astype(bool)

merged["Opponent Difficulty"] = np.where(
    merged["Is Home"],
    merged["team_a_difficulty"],
    merged["team_h_difficulty"]
).astype("Int64")

# ------------------------------------------
# 4️⃣ Map team names (teams.csv)
# ------------------------------------------
def load_team_names_by_season():
    lookup = {}
    for season in sorted(os.listdir(DATA_ROOT)):
        path = Path(DATA_ROOT)/season/"teams.csv"
        if path.exists():
            t = pd.read_csv(path)
            mapping = dict(zip(t["id"], t["name"]))
            lookup[season] = mapping
    return lookup

team_name_map = load_team_names_by_season()

def name_from_id(row, col):
    return team_name_map.get(row["season"], {}).get(row[col], None)

merged["Player Team Name"] = merged.apply(lambda r: name_from_id(r, "Player Team ID"), axis=1)
merged["Opponent Name"]     = merged.apply(lambda r: name_from_id(r, "Opponent ID"), axis=1)

# ------------------------------------------
# 5️⃣ Select final columns
# ------------------------------------------
core_cols = [
    "season","Gameweek","Player Name","element","Player Team ID","Player Team Name",
    "Opponent ID","Opponent Name","Is Home","Opponent Difficulty",
    "minutes","goals_scored","assists","clean_sheets","goals_conceded",
    "yellow_cards","red_cards","bonus","total_points","influence",
    "creativity","threat","ict_index"
]

existing = [c for c in core_cols if c in merged.columns]

df_core = merged[existing].copy()
df_core.rename(columns={"element":"Code"}, inplace=True)

print("\n✅ NEW core table built!")
print("Rows:", len(df_core))
display(df_core.head(10))


📦 Building core training table using (season, GW, team vs opponent) matching...


NameError: name 'fixtures' is not defined

In [ ]:
df_core.head()
df_core["Opponent Difficulty"].isna().sum()


In [ ]:
# %% [markdown]
# ## 4B. Check missing Opponent Difficulty after fixture-based join

# %%
print("🔍 Opponent Difficulty — overall summary")
total_rows = len(df_core)
missing_od = df_core["Opponent Difficulty"].isna().sum()
print(f"  Total rows: {total_rows:,}")
print(f"  Missing Opponent Difficulty: {missing_od:,} ({missing_od/total_rows*100:.2f}%)")

print("\n🔍 Missing Opponent Difficulty by season:")
season_stats = (
    df_core
    .groupby("season")["Opponent Difficulty"]
    .apply(lambda s: s.isna().sum())
    .to_frame("Missing_OD")
)
season_stats["Total_Rows"] = df_core.groupby("season")["Opponent Difficulty"].size()
season_stats["Missing_%"] = (season_stats["Missing_OD"] / season_stats["Total_Rows"] * 100).round(2)

display(season_stats)


In [ ]:
# ## 5. Load fixtures (difficulty table)
def load_fixtures_long():
    rows = []

    for season in sorted(df_core["season"].unique()):
        fx_path = DATA_ROOT / season / "fixtures.csv"
        if not fx_path.exists():
            continue

        fx = pd.read_csv(fx_path)
        fx["Gameweek"] = fx["event"] if "event" in fx.columns else fx.get("round")

        for _, r in fx.iterrows():
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_h"),
                "Opponent ID": r.get("team_a"),
                "Is Home": True,
                "Difficulty": r.get("team_h_difficulty", np.nan)
            })
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_a"),
                "Opponent ID": r.get("team_h"),
                "Is Home": False,
                "Difficulty": r.get("team_a_difficulty", np.nan)
            })

    fixture_long = pd.DataFrame(rows)
    fixture_long["Gameweek"] = pd.to_numeric(fixture_long["Gameweek"], errors="coerce").astype("Int64")
    fixture_long["Team ID"] = pd.to_numeric(fixture_long["Team ID"], errors="coerce").astype("Int64")
    fixture_long["Opponent ID"] = pd.to_numeric(fixture_long["Opponent ID"], errors="coerce").astype("Int64")
    fixture_long["Is Home"] = fixture_long["Is Home"].astype(bool)

    print("Fixtures rows:", len(fixture_long))
    return fixture_long


fixture_long = load_fixtures_long()
fixture_long.head()


Fixtures rows: 5320


,season,Gameweek,Team ID,Opponent ID,Is Home,Difficulty
0,2018-19,1,14,11,True,3
1,2018-19,1,11,14,False,4
2,2018-19,1,15,17,True,4
3,2018-19,1,17,15,False,3
4,2018-19,1,2,5,True,2


In [ ]:
# %% [markdown]
# ## 6. Merge Opponent Difficulty

# %%
df_core["Gameweek"] = pd.to_numeric(df_core["Gameweek"], errors="coerce").astype("Int64")
df_core["Player Team ID"] = pd.to_numeric(df_core["Player Team ID"], errors="coerce").astype("Int64")
df_core["Opponent ID"] = pd.to_numeric(df_core["Opponent ID"], errors="coerce").astype("Int64")

df_core = df_core.merge(
    fixture_long,
    left_on=["season","Gameweek","Player Team ID","Opponent ID","Is Home"],
    right_on=["season","Gameweek","Team ID","Opponent ID","Is Home"],
    how="left"
)

df_core.rename(columns={"Difficulty":"Opponent Difficulty"}, inplace=True)
df_core.drop(columns=["Team ID"], inplace=True)

print("Filled Opponent Difficulty:",
      df_core["Opponent Difficulty"].notna().sum())
df_core.head()

Filled Opponent Difficulty: 170656


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,...,first_name,second_name,Player Team Name,Opponent ID,Opponent Name,short_name,Is Home,Player Name,Web Name,Opponent Difficulty
0,Aaron_Cresswell_402,402,0,0,0,0,0,0,0,0,...,Aaron,Cresswell,West Ham,12,Liverpool,Liverpool,False,Aaron Cresswell,Cresswell,5.0
1,Aaron_Lennon_83,83,90,0,0,1,0,0,0,3,...,Aaron,Lennon,Burnley,16,Southampton,Southampton,False,Aaron Lennon,Lennon,2.0
2,Aaron_Mooy_199,199,90,0,0,0,3,0,0,2,...,Aaron,Mooy,Huddersfield,6,Chelsea,Chelsea,True,Aaron Mooy,Mooy,4.0
3,Aaron_Ramsey_14,14,53,0,0,0,1,0,0,1,...,Aaron,Ramsey,Arsenal,13,Man City,Man City,True,Aaron Ramsey,Ramsey,4.0
4,Aaron_Wan-Bissaka_145,145,90,0,1,1,0,0,0,12,...,Aaron,Wan-Bissaka,Crystal Palace,9,Fulham,Fulham,False,Aaron Wan-Bissaka,Wan-Bissaka,2.0


In [ ]:
# %% [markdown]
# ## 7. Cleaning & Canonical Columns

# %%
POS_MAP = {1:"GK",2:"DEF",3:"MID",4:"FWD"}

rename_map = {
    "element":"Code",
    "minutes":"Minutes Played",
    "goals_scored":"Goals Scored",
    "assists":"Assists",
    "clean_sheets":"Clean Sheet",
    "goals_conceded":"Goals Conceded",
    "yellow_cards":"Yellow Card",
    "red_cards":"Red Cards",
    "total_points":"Total Points",
    "influence":"Influence",
    "creativity":"Creativity",
    "threat":"Threat",
    "ict_index":"ICT Index"
}

df_clean = df_core.copy()
df_clean.rename(columns=rename_map, inplace=True)

df_clean["Position"] = df_clean["element_type"].map(POS_MAP)
df_clean["Player Name Norm"] = df_clean["Player Name"].apply(normalize_player_name)

print("Sample cleaned:")
df_clean[["Player Name","Player Team Name","Opponent Name","Total Points"]].head()


Sample cleaned:


,Player Name,Player Team Name,Opponent Name,Total Points
0,Aaron Cresswell,West Ham,Liverpool,0
1,Aaron Lennon,Burnley,Southampton,3
2,Aaron Mooy,Huddersfield,Chelsea,2
3,Aaron Ramsey,Arsenal,Man City,1
4,Aaron Wan-Bissaka,Crystal Palace,Fulham,12


In [ ]:
# %% [markdown]
# ## 8. Injury flag (3+ games with 0 min)

# %%
df_clean = df_clean.sort_values(["Player Name Norm","season","Gameweek"])
df_clean["Injury/Unavailable"] = 0

for p in df_clean["Player Name Norm"].unique():
    mask = df_clean["Player Name Norm"] == p
    m = df_clean.loc[mask,"Minutes Played"]

    streak=0
    flags=[]
    for x in m:
        if x==0:
            streak+=1
            flags.append(1 if streak>=3 else 0)
        else:
            streak=0
            flags.append(0)
    df_clean.loc[mask,"Injury/Unavailable"] = flags

df_clean.head()


,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Opponent ID,Opponent Name,short_name,Is Home,Player Name,Web Name,Opponent Difficulty,Position,Player Name Norm,Injury/Unavailable
161635,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,5,Brighton,BHA,False,Aaron Anselmino,Anselmino,3.0,DEF,aaron anselmino,0
162428,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,2,Aston Villa,AVL,False,Aaron Anselmino,Anselmino,4.0,DEF,aaron anselmino,0
163211,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,17,Southampton,SOU,True,Aaron Anselmino,Anselmino,1.0,DEF,aaron anselmino,1
163999,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,11,Leicester,LEI,True,Aaron Anselmino,Anselmino,1.0,DEF,aaron anselmino,1
164657,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,1,Arsenal,ARS,False,Aaron Anselmino,Anselmino,5.0,DEF,aaron anselmino,1


In [ ]:
# %% [markdown]
# ## 9. Rolling averages L3 & L5

# %%
def add_lagged(df):
    metrics = [
        "Total Points","Minutes Played","Goals Scored","Assists",
        "Goals Conceded","ICT Index","Threat","Creativity","Influence"
    ]

    df = df.sort_values(["Player Name Norm","season","Gameweek"])

    for w in [3,5]:
        for c in metrics:
            if c not in df.columns:
                continue
            new=f"Avg_{c}_L{w}"
            df[new] = (
                df.groupby(["Player Name Norm","season"])[c]
                .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
                .fillna(0)
            )
    return df

df_lagged = add_lagged(df_clean)
df_lagged.head()


,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
161635,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162428,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
163211,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
163999,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
164657,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# %% [markdown]
# ## 10. UUID mapping (cleaned if exists → used)

# %%
if CLEANED_UUID_MAPPING_PATH.exists():
    print("📥 Using cleaned UUID mapping:", CLEANED_UUID_MAPPING_PATH)
    mapping = pd.read_csv(CLEANED_UUID_MAPPING_PATH)

    norm_col = [c for c in mapping.columns if c.lower()=="player name norm"][0]
    uuid_col = [c for c in mapping.columns if c.lower()=="player uuid"][0]

    uuid_map = dict(zip(mapping[norm_col], mapping[uuid_col]))

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(uuid_map)

    print("UUIDs filled:", df_lagged["Player UUID"].notna().sum())

else:
    print("⚠️ No cleaned mapping. Creating NEW stable mapping...")

    unique_norms = sorted(df_lagged["Player Name Norm"].unique())
    new_uuids = [str(uuid.uuid4()) for _ in unique_norms]

    mapping = pd.DataFrame({
        "Player Name Norm": unique_norms,
        "Player UUID": new_uuids
    })

    rep = df_lagged.groupby("Player Name Norm")[["Player Name","Web Name"]]\
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan).reset_index()

    mapping = mapping.merge(rep, on="Player Name Norm", how="left")

    mapping.to_csv(UUID_MAPPING_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved new UUID mapping → {UUID_MAPPING_PATH}")

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(
        dict(zip(mapping["Player Name Norm"], mapping["Player UUID"]))
    )

df_lagged.head()


📥 Using cleaned UUID mapping: output\player_uuid_mapping_cleaned.csv
UUIDs filled: 170997


,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5,Player UUID
161635,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16b72858-75e4-4125-ad3b-e13f81e6d815
162428,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16b72858-75e4-4125-ad3b-e13f81e6d815
163211,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16b72858-75e4-4125-ad3b-e13f81e6d815
163999,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16b72858-75e4-4125-ad3b-e13f81e6d815
164657,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16b72858-75e4-4125-ad3b-e13f81e6d815


In [ ]:
# %% [markdown]
# ## 11. Final save → training_data_v2.csv

# %%
base_cols = [
    "Player UUID","Code","Player Name","Web Name","Player Team Name",
    "season","Gameweek","Minutes Played","Goals Scored","Assists",
    "Clean Sheet","Goals Conceded","Yellow Card","Red Cards",
    "Total Points","Threat","ICT Index","Influence","Creativity",
    "Opponent Name","Opponent Difficulty","Is Home","Position","Injury/Unavailable"
]

lagged_cols = [c for c in df_lagged.columns if c.startswith("Avg_")]
final_cols = base_cols + lagged_cols

df_final = df_lagged[final_cols].copy()

df_final.to_csv(TRAINING_PATH, index=False, encoding="utf-8-sig")

print("🎉 training_data_v2.csv CREATED!")
print("Rows:", len(df_final))
print("Cols:", len(df_final.columns))
df_final.head()


🎉 training_data_v2.csv CREATED!
Rows: 171993
Cols: 42


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
161635,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162428,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
163211,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
163999,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
164657,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# %% [markdown]
# ## 12. Sanity Check

# %%
print("Unique seasons:", sorted(df_final["season"].unique()))
print("Max GW per season:", df_final.groupby("season")["Gameweek"].max().to_dict())

print("Opponent Difficulty non-null:", df_final["Opponent Difficulty"].notna().sum())
df_final[
    ["Player Name","Player Team Name","Opponent Name","season","Gameweek","Opponent Difficulty"]
].head(20)


Unique seasons: ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Max GW per season: {'2018-19': 38, '2019-20': 29, '2020-21': 38, '2021-22': 38, '2022-23': 38, '2023-24': 38, '2024-25': 38}
Opponent Difficulty non-null: 170656


,Player Name,Player Team Name,Opponent Name,season,Gameweek,Opponent Difficulty
161635,Aaron Anselmino,Chelsea,Brighton,2024-25,25,3.0
162428,Aaron Anselmino,Chelsea,Aston Villa,2024-25,26,4.0
163211,Aaron Anselmino,Chelsea,Southampton,2024-25,27,1.0
163999,Aaron Anselmino,Chelsea,Leicester,2024-25,28,1.0
164657,Aaron Anselmino,Chelsea,Arsenal,2024-25,29,5.0
165429,Aaron Anselmino,Chelsea,Spurs,2024-25,30,2.0
166221,Aaron Anselmino,Chelsea,Brentford,2024-25,31,3.0
167079,Aaron Anselmino,Chelsea,Ipswich,2024-25,32,2.0
168021,Aaron Anselmino,Chelsea,Fulham,2024-25,33,3.0
168706,Aaron Anselmino,Chelsea,Everton,2024-25,34,3.0


In [ ]:
print(df_gws['season'].unique())
print(fixture_long['season'].unique())


['2018-19' '2019-20' '2020-21' '2021-22' '2022-23' '2023-24' '2024-25'
 '2025-26']
['2018-19' '2019-20' '2020-21' '2021-22' '2022-23' '2023-24' '2024-25'
 '2025-26']


In [ ]:
print(sorted(df_gws['Gameweek'].unique())[:10])
print(sorted(fixture_long['Gameweek'].unique())[:10])


[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [ ]:
# ## 8A. Diagnostic: Missing Opponent Difficulty per season

print("🔍 Opponent Difficulty — overall summary")
total_rows = len(df_core)
missing_od = df_core["Opponent Difficulty"].isna().sum()
print(f"  Total rows: {total_rows:,}")
print(f"  Missing Opponent Difficulty: {missing_od:,} ({missing_od/total_rows*100:.2f}%)")

print("\n🔍 Missing Opponent Difficulty by season:")
season_stats = (
    df_core
    .groupby("season")["Opponent Difficulty"]
    .apply(lambda s: s.isna().sum())
    .to_frame("Missing_OD")
)

season_stats["Total_Rows"] = df_core.groupby("season")["Opponent Difficulty"].size()
season_stats["Missing_%"] = (season_stats["Missing_OD"] / season_stats["Total_Rows"] * 100).round(2)

display(season_stats)

🔍 Opponent Difficulty — overall summary
  Total rows: 171,993
  Missing Opponent Difficulty: 1,337 (0.78%)

🔍 Missing Opponent Difficulty by season:


,Missing_OD,Total_Rows,Missing_%
season,,,
2018-19,85,21790,0.39
2019-20,187,16556,1.13
2020-21,138,24365,0.57
2021-22,189,25447,0.74
2022-23,307,26505,1.16
2023-24,175,29725,0.59
2024-25,256,27605,0.93


In [ ]:

# ## 8B. Diagnostic: Are we missing IDs or Is Home?

missing = df_core[df_core["Opponent Difficulty"].isna()].copy()

print(f"🔍 Rows with missing Opponent Difficulty: {len(missing):,}")

print("\n👉 Πόσες από αυτές έχουν κενά στα keys του join:")
for col in ["Player Team ID", "Opponent ID", "Is Home", "Gameweek"]:
    nulls = missing[col].isna().sum()
    print(f"  {col:15s}: {nulls:6d} ({nulls/len(missing)*100:5.2f}%)")


🔍 Rows with missing Opponent Difficulty: 1,337

👉 Πόσες από αυτές έχουν κενά στα keys του join:
  Player Team ID :      0 ( 0.00%)
  Opponent ID    :      0 ( 0.00%)
  Is Home        :      0 ( 0.00%)
  Gameweek       :      0 ( 0.00%)


In [ ]:
# %% [markdown]
# ## 8C. Diagnostic: Match vs fixtures (anti-join)

# %%
# Ετοιμάζουμε fixtures με ίδια ονόματα keys όπως στο df_core
fixture_keys = fixture_long.rename(columns={"Team ID": "Player Team ID"}).copy()

keys = ["season", "Gameweek", "Player Team ID", "Opponent ID", "Is Home"]

missing_diag = (
    df_core[df_core["Opponent Difficulty"].isna()]
    .merge(
        fixture_keys[keys + ["Difficulty"]],
        on=keys,
        how="left",
        indicator=True,
        suffixes=("", "_fx")
    )
)

print("🔍 Merge indicator counts:")
print(missing_diag["_merge"].value_counts())

print("\nℹ️ Ερμηνεία:")
print("  'left_only'   → δεν βρέθηκε ΚΑΘΟΛΟΥ αντίστοιχο fixture με αυτά τα keys.")
print("  'both'        → βρέθηκε fixture αλλά το Difficulty είναι πιθανότατα NaN στο fixtures.csv.\n")

print("🔍 Για τις γραμμές που ταίριαξαν (both), πόσα έχουν Difficulty από fixtures;")
matched = missing_diag[missing_diag["_merge"] == "both"]
if not matched.empty:
    filled_from_fx = matched["Difficulty"].notna().sum()
    print(f"  Matched rows: {len(matched):,}")
    print(f"  με μη-NaN Difficulty στο fixtures: {filled_from_fx:,} ({filled_from_fx/len(matched)*100:.2f}%)")
else:
    print("  (Καμία γραμμή με _merge == 'both')")

print("\n🔍 Δείγμα από περιπτώσεις που ΔΕΝ βρέθηκαν καθόλου στα fixtures (left_only):")
sample_left = missing_diag[missing_diag["_merge"] == "left_only"].head(10)
display(sample_left[[
    "season","Gameweek","Player Team ID","Opponent ID","Is Home",
    "Player Name","Player Team Name","Opponent Name"
]])


🔍 Merge indicator counts:
_merge
left_only     1337
right_only       0
both             0
Name: count, dtype: int64

ℹ️ Ερμηνεία:
  'left_only'   → δεν βρέθηκε ΚΑΘΟΛΟΥ αντίστοιχο fixture με αυτά τα keys.
  'both'        → βρέθηκε fixture αλλά το Difficulty είναι πιθανότατα NaN στο fixtures.csv.

🔍 Για τις γραμμές που ταίριαξαν (both), πόσα έχουν Difficulty από fixtures;
  (Καμία γραμμή με _merge == 'both')

🔍 Δείγμα από περιπτώσεις που ΔΕΝ βρέθηκαν καθόλου στα fixtures (left_only):


,season,Gameweek,Player Team ID,Opponent ID,Is Home,Player Name,Player Team Name,Opponent Name
0,2018-19,1,2,19,True,Dominic Solanke,Bournemouth,West Ham
1,2018-19,1,10,9,False,Jason Puncheon,Huddersfield,Fulham
2,2018-19,1,2,19,True,Nathaniel Clyne,Bournemouth,West Ham
3,2018-19,1,5,20,False,Oumar Niasse,Cardiff,Wolves
4,2018-19,2,2,7,False,Dominic Solanke,Bournemouth,Crystal Palace
5,2018-19,2,10,12,True,Jason Puncheon,Huddersfield,Liverpool
6,2018-19,2,2,7,False,Nathaniel Clyne,Bournemouth,Crystal Palace
7,2018-19,2,5,16,True,Oumar Niasse,Cardiff,Southampton
8,2018-19,3,2,3,True,Dominic Solanke,Bournemouth,Brighton
9,2018-19,3,10,18,False,Jason Puncheon,Huddersfield,Watford


In [ ]:
# %% [markdown]
# ## 8D. Diagnostic: Are fixtures missing difficulty by season?

# %%
fx_stats = (
    fixture_long
    .assign(DiffMissing = fixture_long["Difficulty"].isna())
    .groupby("season")["DiffMissing"]
    .agg(["sum","count"])
    .rename(columns={"sum":"Missing_Difficulty","count":"Total_Fixture_Rows"})
)
fx_stats["Missing_%"] = (fx_stats["Missing_Difficulty"] / fx_stats["Total_Fixture_Rows"] * 100).round(2)

print("🔍 Difficulty presence inside fixtures:")
display(fx_stats)


🔍 Difficulty presence inside fixtures:


,Missing_Difficulty,Total_Fixture_Rows,Missing_%
season,,,
2018-19,0,760,0.0
2019-20,0,760,0.0
2020-21,0,760,0.0
2021-22,0,760,0.0
2022-23,0,760,0.0
2023-24,0,760,0.0
2024-25,0,760,0.0
